#### Embeddings mit gemini-embedding-001 kreieren

In [ ]:
from google import genai

client = genai.Client()

texte =[
        "What is the meaning of life?",
        "What is the purpose of existence?",
        "How do I bake a cake?"
]

result = client.models.embed_content(
        model="gemini-embedding-001",
        contents= texte
)

for embedding in result.embeddings:
    print(embedding)
    print("-" * 40)

#### Embeddings mit OpenAI's Modell text-embedding-3-small kreieren

In [ ]:
from openai import OpenAI

client = OpenAI()  # holt sich den OPENAI_API_KEY in der Umgebung (.env - Datei!)

texte = [
        "What is the meaning of life?",
        "What is the purpose of existence?",
        "How do I bake a cake?"
]

result = client.embeddings.create(
        model="text-embedding-3-small",
        input=texte,
)

vektoren = [item.embedding for item in result.data]

for vektor in vektoren:
    print(f"Vektor-Länge: {len(vektor)}")
    print(vektor)
    print("-" * 40)

Dataframe mit Ähnlichkeitsvergleich (Auf Basis der Gemini-Embeddings)

In [ ]:
from pathlib import Path
from google import genai
from google.genai import types
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

client = genai.Client()

EMBEDDING_MODEL = "gemini-embedding-001"
SPEICHERDATEI = Path("embeddings.json")

texts = [
    "What is the meaning of life?",
    "What is the purpose of existence?",
    "How do I bake a cake?",
]

result = client.models.embed_content(
    model="gemini-embedding-001",
    contents=texts,
    config=types.EmbedContentConfig(task_type="SEMANTIC_SIMILARITY")
)

# Create a 3x3 table to show the similarity matrix
df = pd.DataFrame(
    cosine_similarity([e.values for e in result.embeddings]),
    index=texts,
    columns=texts,
)

df

### Einfaches Programm zur Ähnlichkeitssuche (mit JSON-Vektordatenbank)

mit Gemini:

In [16]:
from __future__ import annotations

import json
import math
from pathlib import Path
from typing import List, Dict, Any

from google import genai
from google.genai import types

client = genai.Client()  # erwartet GEMINI_API_KEY oder GOOGLE_API_KEY in der Umgebung

EMBEDDING_MODEL = "gemini-embedding-001"
SPEICHERDATEI = Path("gemini_embeddings.json")


def get_embeddings(texts: List[str]) -> List[List[float]]:
    """Erzeugt Embeddings für mehrere Texte mit Gemini."""
    result = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=texts,
        config=types.EmbedContentConfig(
            task_type="RETRIEVAL_QUERY"     # hier bei gemini gibt es verschiedene Modi (SEMANTIC_SIMILARITY, RETRIEVAL_QUERY, FACT_VERIFICATION etc.)
        ),
    )
    return [embedding.values for embedding in result.embeddings]


def cosine_similarity(a: List[float], b: List[float]) -> float:
    """Berechnet die Cosine Similarity zwischen zwei Vektoren."""
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(y * y for y in b))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)


def save_embeddings(texts: List[str], datei: Path = SPEICHERDATEI) -> None:
    """Erzeugt Embeddings und speichert Text + Vektor als JSON."""
    vectors = get_embeddings(texts)

    daten: List[Dict[str, Any]] = [
        {"text": text, "embedding": vector}
        for text, vector in zip(texts, vectors)
    ]

    with datei.open("w", encoding="utf-8") as f:
        json.dump(daten, f, ensure_ascii=False, indent=2)


def load_embeddings(datei: Path = SPEICHERDATEI) -> List[Dict[str, Any]]:
    """Lädt gespeicherte Embeddings aus JSON."""
    with datei.open("r", encoding="utf-8") as f:
        return json.load(f)


def suche_aehnliche_texte(
    query: str,
    daten: List[Dict[str, Any]],
    top_k: int = 3,
) -> List[Dict[str, Any]]:
    """Vektorisiert den Query und gibt die ähnlichsten Texte zurück."""
    query_embedding = get_embeddings([query])[0]

    ergebnisse = []
    for eintrag in daten:
        score = cosine_similarity(query_embedding, eintrag["embedding"])
        ergebnisse.append(
            {
                "text": eintrag["text"],
                "score": score,
            }
        )

    ergebnisse.sort(key=lambda x: x["score"], reverse=True)
    return ergebnisse[:top_k]

In [19]:

texte = [
    "Die Vertragsfreiheit ist ein zentraler Grundsatz des Zivilrechts.",
    "Im Verwaltungsrecht spielt der Verhältnismäßigkeitsgrundsatz eine wichtige Rolle.",
    "Eine Anfechtung wegen Irrtums richtet sich nach den Regeln des BGB.",
    "Der Staat darf nur auf gesetzlicher Grundlage in Grundrechte eingreifen.",
    "Der Himmel ist blau und die Sonne scheint."
]
query = "Nach welchen Regeln kann ein Vertrag angefochten werden?"

In [ ]:

# Nur beim ersten Mal ausführen, um die Embeddings zu erzeugen:
save_embeddings(texte)

daten = load_embeddings()

treffer = suche_aehnliche_texte(query, daten, top_k=5)

print(f"\nQuery: {query}\n")
for i, treffer_eintrag in enumerate(treffer, start=1):
    print(f"{i}. Score: {treffer_eintrag['score']:.4f}")
    print(f"   Text: {treffer_eintrag['text']}")

mit OpenAI:

In [20]:
from __future__ import annotations

import json
import math
from pathlib import Path
from typing import List, Dict, Any

from openai import OpenAI

client = OpenAI()  # erwartet OPENAI_API_KEY in der Umgebung


EMBEDDING_MODEL = "text-embedding-3-small"
SPEICHERDATEI = Path("openai_embeddings.json")


def get_embeddings(texts: List[str]) -> List[List[float]]:
    """Erzeugt Embeddings für mehrere Texte."""
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=texts,
    )
    return [item.embedding for item in response.data]


def cosine_similarity(a: List[float], b: List[float]) -> float:
    """Berechnet die Cosine Similarity zwischen zwei Vektoren."""
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(y * y for y in b))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)


def speichere_embeddings(texts: List[str], datei: Path = SPEICHERDATEI) -> None:
    """Erzeugt Embeddings und speichert Text + Vektor als JSON."""
    vectors = get_embeddings(texts)

    daten: List[Dict[str, Any]] = [
        {"text": text, "embedding": vector}
        for text, vector in zip(texts, vectors)
    ]

    with datei.open("w", encoding="utf-8") as f:
        json.dump(daten, f, ensure_ascii=False, indent=2)


def lade_embeddings(datei: Path = SPEICHERDATEI) -> List[Dict[str, Any]]:
    """Lädt gespeicherte Embeddings aus JSON."""
    with datei.open("r", encoding="utf-8") as f:
        return json.load(f)


def suche_aehnliche_texte(
    query: str,
    daten: List[Dict[str, Any]],
    top_k: int = 3,
) -> List[Dict[str, Any]]:
    """Vektorisiert den Query und gibt die ähnlichsten Texte zurück."""
    query_embedding = get_embeddings([query])[0]

    ergebnisse = []
    for eintrag in daten:
        score = cosine_similarity(query_embedding, eintrag["embedding"])
        ergebnisse.append(
            {
                "text": eintrag["text"],
                "score": score,
            }
        )

    ergebnisse.sort(key=lambda x: x["score"], reverse=True)
    return ergebnisse[:top_k]



In [21]:
texte = [
    "Die Vertragsfreiheit ist ein zentraler Grundsatz des Zivilrechts.",
    "Im Verwaltungsrecht spielt der Verhältnismäßigkeitsgrundsatz eine wichtige Rolle.",
    "Eine Anfechtung wegen Irrtums richtet sich nach den Regeln des BGB.",
    "Der Staat darf nur auf gesetzlicher Grundlage in Grundrechte eingreifen.",
]
query = "Nach welchen Regeln kann ein Vertrag angefochten werden?"


In [ ]:
# 1. Embeddings erzeugen und speichern (auskommentiert, damit es nicht jedes Mal neu gemacht wird)
speichere_embeddings(texte)

# 2. Embeddings wieder laden
daten = lade_embeddings()

# 3. Ähnlichkeitssuche
treffer = suche_aehnliche_texte(query, daten, top_k=3)

print(f"\nQuery: {query}\n")
for i, treffer_eintrag in enumerate(treffer, start=1):
    print(f"{i}. Score: {treffer_eintrag['score']:.4f}")
    print(f"   Text: {treffer_eintrag['text']}")